1. **Load**

In [40]:
import re
import unicodedata
import pandas as pd
import spacy

from pathlib import Path
from spacy.language import Language
from spacy.matcher import PhraseMatcher
from spacy.tokens import Span
from negspacy.negation import Negex
from negspacy.termsets import termset
from scispacy.linking import EntityLinker

fyp_dir = Path.cwd().parent
file_path = fyp_dir / "data"

train_df = pd.read_csv(f"{file_path}/train_split.csv")
test_df  = pd.read_csv(f"{file_path}/test_split.csv")

# train_df.head()

2. **Cleansing**

In [ ]:
ABBR_MAP = {
    "cp": "chest pain",
    "c/p": "chest pain",

    "sob": "shortness of breath",
    "s.o.b": "shortness of breath",
    "soa": "shortness of breath",
    "s/p": "status post",

    "n/v": "nausea vomiting",
    "n/vom": "nausea vomiting",
    "n/vomiting": "nausea vomiting",
    "n/v/d": "nausea vomiting diarrhea",

    "abd pain": "abdominal pain",
    "ap": "abdominal pain",
    "ah": "auditory hallucinations",

    "ha": "headache",
    "h/a": "headache",

    "diz": "dizziness",
    "loc": "loss of consciousness",
    "syn": "syncope",
    "faint": "syncope",

    "uti": "urinary tract infection",
    "uti.": "urinary tract infection",
    "uti/s": "urinary tract infection",
    "uti?": "urinary tract infection",

    "hematuria": "hematuria",

    "etoh": "ethanol",
    "vh": "visual hallucinations",

    "ili": "influenza-like illness",
}

def normalize_abbr(text):
    if text is None or pd.isna(text):
        return ""

    text = unicodedata.normalize("NFKC", str(text))
    text = text.lower().strip()

    # Normalize common punctuation variants.
    text = text.replace("’", "'")
    text = text.replace("(", " ")
    text = text.replace(")", " ")
    text = text.replace(",", " ")
    text = text.replace(";", " ")
    text = text.replace("|", " ")

    # Normalize repeated whitespace before abbreviation matching.
    text = re.sub(r"\s+", " ", text).strip()

    # Match longest abbreviations first.
    for abbreviation, expansion in sorted(
        ABBR_MAP.items(),
        key=lambda item: len(item[0]),
        reverse=True,
    ):
        pattern = rf"(?<!\w){re.escape(abbreviation)}(?!\w)"
        text = re.sub(pattern, expansion, text)

    # Remove unnecessary trailing punctuation.
    text = re.sub(r"[.!?]+$", "", text)

    # Collapse whitespace introduced by replacements.
    text = re.sub(r"\s+", " ", text).strip()

    return text

def clean_text(value):
    return normalize_abbr(value)


# # Test case
# ABBR_TESTS = {
#     "CP": "chest pain",
#     "SOB": "shortness of breath",
#     "c/p and n/v": "chest pain and nausea vomiting",
#     "abd pain, denies N/V": (
#         "abdominal pain denies nausea vomiting"
#     ),
#     "UTI?": "urinary tract infection",
# }

# for raw, expected in ABBR_TESTS.items():
#     actual = clean_text(raw)
#     print(f"{raw!r} -> {actual!r}")


------------------------------------------------------------------------

3. **Build a negation-aware spaCy pipeline**


a.) Create a spaCy component for Negex

In [42]:
from negspacy.negation import Negex
from spacy.matcher import PhraseMatcher
from spacy.tokens import Span
from spacy.language import Language

SYMPTOM_KEYWORDS = [
    "chest pain",
    "shortness of breath",
    "dyspnea",
    "cough",
    "fever",
    "vomiting",
    "nausea",
    "abdominal pain",
    "back pain",
    "headache",
    "dizziness",
    "syncope",
    "weakness",
    "numbness",
    "stroke",
    "bleeding",
    "blood in stool",
    "hematemesis",
    "suicidal",
    "overdose",
    "burn",
    "urinary tract infection",
    "dysuria",
    "hematuria",
    "seizure",
    "neck pain",
    "palpitations",
    "diarrhea",
    "dehydration",
]

# Load spacy
nlp = spacy.load("en_core_web_sm", disable=["ner"],)

# Create matcher
matcher = PhraseMatcher(nlp.vocab, attr='LOWER')

patterns = [
    nlp.make_doc(keyword) for keyword in SYMPTOM_KEYWORDS
]

matcher.add("SYMPTOM", patterns)

# Custom Component
@Language.component("keyword_adder")
def keyword_adder(doc):
    matches = matcher(doc)

    spans = [Span(doc, start, end, label="SYMPTOM") for _, start, end in matches]

    # remove overlaps: keep longest spans (simple heuristic)
    spans = sorted(spans, key=lambda s: (s.start, - (s.end - s.start)))

    selected = []
    occupied = set()

    for s in spans:
        token_ids = set(range(s.start, s.end))

        if occupied.isdisjoint(token_ids):
            selected.append(s)
            occupied.update(token_ids)

    doc.ents = selected
    return doc

# Add pipeline 1
if "keyword_adder" in nlp.pipe_names:
    nlp.remove_pipe("keyword_adder")

nlp.add_pipe("keyword_adder", last=True)

# # test case
# text = "The patient has fever but denies chest pain."
# doc = nlp(text)

# for ent in doc.ents:
#     print(ent.text, ent._.negex)

c:\Users\ngngo\AppData\Local\Programs\Python\Python312\Lib\site-packages\spacy\util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


<function __main__.keyword_adder(doc)>

## Configure negation explicitly

In [43]:
"""
The order is important:
text
    -> keyword_adder creates entities
    -> negex assigns negation status
"""

# Pipeline 1
clinical_termset = termset("en_clinical")
negation_patterns = clinical_termset.get_patterns()

# Add local patterns useful for chief complaints.
negation_patterns["preceding_negations"].extend([
    "denies",
    "deny",
    "without",
    "negative for",
    "no",
    "not experiencing",
    "not reporting",
])

negation_patterns["following_negations"].extend([
    "denied",
    "absent",
])

negation_patterns["termination"].extend([
    "but",
    "however",
    "although",
    "except",
])


# Remove duplicates
for key in negation_patterns:
    negation_patterns[key] = list(set(negation_patterns[key]))

# Add negex for pipeline 1
if "negex" in nlp.pipe_names:
    nlp.remove_pipe("negex")

nlp.add_pipe(
    "negex",
    last=True,
    config={
        "ent_types": ["SYMPTOM"],
        "neg_termset": negation_patterns,
        "chunk_prefix": ["no"],
    },
)


# # Test case
# test_text = (
#     "patient denies chest pain and shortness of breath "
#     "no fever but reports dizziness"
# )

# doc = nlp(clean_text(test_text))

# for ent in doc.ents:
#     print(
#         ent.text,
#         "negated =", ent._.negex,
#         "start =", ent.start_char,
#         "end =", ent.end_char,
#     )


## scispaCy-style UMLS linking

In [44]:
# Load scispacy
nlp_umls = spacy.load("en_core_sci_sm")

if "scispacy_linker" in nlp_umls.pipe_names:
    nlp_umls.remove_pipe("scispacy_linker")

# add UMLS linker pipeline 2
nlp_umls.add_pipe(
    "scispacy_linker",
    config={
        "linker_name": "umls",
        "resolve_abbreviations": True,
        "threshold": 0.80,
    }
)

# Add negex to pipeline 2
if "negex" in nlp_umls.pipe_names:
    nlp_umls.remove_pipe("negex")

nlp_umls.add_pipe(
    "negex",
    last=True,
    config={
        "ent_types": ["ENTITY"],   # or remove ent_types if needed
        "neg_termset": negation_patterns,
        "chunk_prefix": ["no"],
    },
)

linker = nlp_umls.get_pipe("scispacy_linker")


c:\Users\ngngo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.1.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\ngngo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.1.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## Custom Symptom_List Pipline

In [45]:
# Extract Evidence (pipeline 1)
def extract_evidence(doc):
    """
    Returns list of dicts:
      {
        "concept": <keyword or matched phrase in lowercase>,
        "negated": bool,
        "span_text": <surface form>,
        "char_start": int,
        "char_end": int
      }
    """

    evidence = []
    for ent in doc.ents:
        negated = bool(ent._.negex)

        evidence.append({
            "concept": ent.text.lower().strip(),
            "negated": negated,
            "span_text": ent.text,
            "char_start": ent.start_char,
            "char_end": ent.end_char,
        })

    return evidence

# Define evidence extract function (pipeline 2)
def umls_link_evidence(doc, linker):
    """
    Returns list of UMLS-linked evidence dicts:
      {
        "concept": <canonical_name in lowercase>,
        "negated": bool,
        "span_text": <surface form>,
        "char_start": int,
        "char_end": int,
      }
    """
    
    evidence = []

    for ent in doc.ents:
        if not ent._.kb_ents:
            continue

        cui, score = ent._.kb_ents[0]
        kb_ent = linker.kb.cui_to_entity[cui]

        evidence.append({
            "concept": kb_ent.canonical_name.lower().strip(),
            "negated": bool(ent._.negex),
            "span_text": ent.text,
            "char_start": ent.start_char,
            "char_end": ent.end_char,
        })

    return evidence

## Combine two pipelines into one pipeline

In [46]:
# Preserve order if needed
def unique_preserve_order(values):
    seen = set()
    output = []

    for value in values:
        value = value.lower().strip()

        if value and value not in seen:
            output.append(value)
            seen.add(value)

    return output

# single-text preprocessing function
def preprocess_single(text: str, linker):
    normalized_text = clean_text(text)
    doc = nlp(normalized_text)
    doc_umls = nlp_umls(normalized_text)

    keyword_evidence = extract_evidence(doc)
    umls_evidence = umls_link_evidence(doc_umls, linker)

    evidence = keyword_evidence + umls_evidence

    affirmed = unique_preserve_order([
        item["concept"]
        for item in evidence
        if not item["negated"]
    ])

    negated = unique_preserve_order([
        item["concept"]
        for item in evidence
        if item["negated"]
    ])

    return {
        "text": normalized_text,
        "evidence": evidence,
        "affirmed_concepts": affirmed,
        "negated_concepts": negated,
    }

------------------------------------------------------------------------

## Testing

In [47]:
# result = preprocess_single(
#     "chest pain but denies shortness of breath"
# , linker)
# result

------------------------------------------------------------------------

## Conduct to Train and Test


In [52]:
# Define apply function
def preprocess_dataframe(
    df,
    source_col="chiefcomplaint",
    batch_size=512,
    verbose=True,
):
    out = df.copy()
    texts = out[source_col].fillna("").map(clean_text).tolist()
    n = len(texts)

    results = []

    kw_pipe = nlp.pipe(texts, batch_size=batch_size)
    umls_pipe = nlp_umls.pipe(texts, batch_size=batch_size)

    for i, (text, doc_kw, doc_umls) in enumerate(zip(texts, kw_pipe, umls_pipe), start=1):
        if verbose and (i == 1 or i % 5000 == 0 or i == n):
            print(f"Processing {i}/{n} rows...")

        kw_evidence = extract_evidence(doc_kw)
        umls_evidence_items = umls_link_evidence(doc_umls, linker)

        evidence = kw_evidence + umls_evidence_items

        seen = set()
        deduped_evidence = []
        for item in evidence:
            key = (
                item["concept"].lower().strip(),
                item["span_text"].lower().strip(),
                item["char_start"],
                item["char_end"],
                item["negated"],
            )
            if key not in seen:
                deduped_evidence.append(item)
                seen.add(key)

        affirmed = unique_preserve_order(
            item["concept"] for item in deduped_evidence if not item["negated"]
        )
        negated = unique_preserve_order(
            item["concept"] for item in deduped_evidence if item["negated"]
        )

        results.append({
            "text": text,
            "evidence": deduped_evidence,
            "affirmed_concepts": affirmed,
            "negated_concepts": negated,
        })

    result_df = pd.DataFrame(results, index=out.index)

    out["text"] = result_df["text"]
    out["evidence"] = result_df["evidence"]
    out["affirmed_concepts"] = result_df["affirmed_concepts"]
    out["negated_concepts"] = result_df["negated_concepts"]

    if verbose:
        print("Done.")

    return out

# Run it
train_processed = preprocess_dataframe(train_df)
test_processed = preprocess_dataframe(test_df)

Processing 1/334466 rows...
Processing 5000/334466 rows...
Processing 10000/334466 rows...
Processing 15000/334466 rows...
Processing 20000/334466 rows...
Processing 25000/334466 rows...
Processing 30000/334466 rows...
Processing 35000/334466 rows...
Processing 40000/334466 rows...
Processing 45000/334466 rows...
Processing 50000/334466 rows...
Processing 55000/334466 rows...
Processing 60000/334466 rows...
Processing 65000/334466 rows...
Processing 70000/334466 rows...
Processing 75000/334466 rows...
Processing 80000/334466 rows...
Processing 85000/334466 rows...
Processing 90000/334466 rows...
Processing 95000/334466 rows...
Processing 100000/334466 rows...
Processing 105000/334466 rows...
Processing 110000/334466 rows...
Processing 115000/334466 rows...
Processing 120000/334466 rows...
Processing 125000/334466 rows...
Processing 130000/334466 rows...
Processing 135000/334466 rows...
Processing 140000/334466 rows...
Processing 145000/334466 rows...
Processing 150000/334466 rows...
Pr

In [53]:
train_processed.to_csv(f"{fyp_dir}/data/train_processed.csv", index=False)
test_processed.to_csv(f"{fyp_dir}/data/test_processed.csv", index=False)

print("Saved: train_processed.csv and test_processed.csv")

Saved: train_processed.csv and test_processed.csv


------------------------------------------------------------------------

## Quality Checks

In [54]:
from collections import Counter

def validate_row(row):
    affirmed = set(row["affirmed_concepts"])
    negated = set(row["negated_concepts"])

    # Mutual Exclusion
    assert affirmed.isdisjoint(negated), (
        "A concept appears as both affirmed and negated."
    )

    evidence_concepts = {
        item["concept"]
        for item in row["evidence"]
    }

    # Evidence Grounding
    assert affirmed.issubset(evidence_concepts)
    assert negated.issubset(evidence_concepts)

    return True


for _, row in test_processed.head(1000).iterrows():
    validate_row(row)

for _, row in train_processed.head(1000).iterrows():
    validate_row(row)

affirmed_counts = Counter(
    concept
    for concepts in train_processed["affirmed_concepts"]
    for concept in concepts
)

negated_counts = Counter(
    concept
    for concepts in train_processed["negated_concepts"]
    for concept in concepts
)

# Frequency Distribution Check
print("Affirmed concepts:")
print(affirmed_counts.most_common(20))

print("\nNegated concepts:")
print(negated_counts.most_common(20))

# Identifying "Dead" Data   
unmatched = train_processed[
    train_processed["affirmed_concepts"].map(len).eq(0) &
    train_processed["negated_concepts"].map(len).eq(0)
]

unmatched[["chiefcomplaint", "text"]].head(50)

Affirmed concepts:
[('abdominal pain', 38654), ('chest pain', 25274), ('dyspnea', 16611), ('nausea', 15884), ('vomiting', 15291), ('back pain', 13261), ('headache', 12241), ('fever', 11094), ('nausea and vomiting', 10146), ('weakness', 9842), ('dizziness', 8018), ('abnormal', 7393), ('cough', 6993), ('diarrhea', 6790), ('ethanol', 5243), ('mental state', 4847), ('syncope', 4737), ('asthenia', 4705), ('flank pain', 4444), ('pain in lower limb', 4179)]

Negated concepts:
[('stroke', 48), ('cerebrovascular accident', 48), ('urinary tract infection', 46), ('pain of nose', 42), ('nose', 23), ('epistaxis', 22), ('ectopic', 21), ('sexually transmitted diseases', 19), ('septicemia', 15), ('injury of nose', 15), ('hypersensitivity', 14), ('pneumonia', 10), ('communicable diseases', 9), ('hemorrhage', 9), ('seizure', 8), ('seizures', 8), ('abscess', 7), ('ectopic pregnancy', 7), ('herpes zoster disease', 7), ('cauda equina', 7)]


,chiefcomplaint,text
32,BILIARY DRAIN MALFUNCTION,biliary drain malfunction
44,Unresponsive,unresponsive
61,LEFT LEG CELLULITIS,left leg cellulitis
69,SI,si
74,"MVC, Transfer",mvc transfer
85,MVC,mvc
96,Chills,chills
110,SI,si
111,MELENA,melena
114,Melena,melena


## Baseline model

In [55]:
"""
The baseline lexicon was constructed from frequent chief complaint concepts in MIMIC-IV-ED and grouped by clinical acuity into High, 
Medium, and Low risk categories. The mapping was manually curated and refined through error analysis.
"""

import ast

# Reload data
train_processed = pd.read_csv(f"{fyp_dir}/data/train_processed.csv")
test_processed = pd.read_csv(f"{fyp_dir}/data/test_processed.csv")

print(train_processed["risk"].value_counts())
print(test_processed["risk"].value_counts())


concept_counts = (
    train_processed["affirmed_concepts"]
    .explode()
    .value_counts()
)

top_50_dict = concept_counts.head(50).to_dict()
print(top_50_dict)

"""
classify by the highest-risk concept present, and keep the lexicon aligned to your actual chief complaints.
"""

HIGH = {
    "chest pain",
    "dyspnea",
    "shortness of breath",
    "syncope",
    "seizure",
    "overdose",
    "bleeding",
    "hematemesis",
    "hematuria",
    "altered mental status",
    "numbness",
    "weakness",
}

MED = {
    "abdominal pain",
    "fever",
    "vomiting",
    "nausea",
    "dizziness",
    "headache",
    "palpitations",
    "dysuria",
    "diarrhea",
    "cough",
    "neck pain",
    "back pain",
    "urinary tract infection",
    "burn",
}

LOW = {
    "cough",
    "back pain",
    "neck pain",
    "dysuria",
    "diarrhea",
}

def parse_concepts(x):
    if pd.isna(x):
        return set()
    
    if isinstance(x, str):
        try:
            x = ast.literal_eval(x)
        except:
            return set()
    return set(x)

def predict_risk(concepts):
    concepts = parse_concepts(concepts)
    if concepts & HIGH:
        return "High"
    
    elif concepts & MED:
        return "Medium"
    
    else:
        return "Low"

train_processed["pred_risk"] = train_processed["affirmed_concepts"].apply(predict_risk)
test_processed["pred_risk"] = test_processed["affirmed_concepts"].apply(predict_risk)

risk
Medium    180048
High      130737
Low        23681
Name: count, dtype: int64
risk
Medium    45012
High      32685
Low        5920
Name: count, dtype: int64
{'[]': 47626, "['abdominal pain']": 23713, "['chest pain']": 16253, "['dyspnea']": 7052, "['headache']": 6582, "['back pain']": 5688, "['fever']": 4445, "['ethanol']": 4223, "['dizziness']": 3391, "['mental state']": 3328, "['syncope']": 3234, "['weakness', 'asthenia']": 3103, "['back pain', 'low back pain']": 3101, "['knee pain']": 2806, "['flank pain']": 2792, "['pain in lower limb']": 2679, "['cough']": 2669, "['nausea', 'vomiting', 'nausea and vomiting']": 2606, "['foot pain']": 2378, "['abnormal', 'laboratory test finding']": 2310, "['pharyngitis']": 2048, "['abdominal pain', 'nausea', 'vomiting', 'nausea and vomiting']": 2023, "['seizure', 'seizures']": 1976, "['neck pain']": 1935, "['hematochezia']": 1923, "['craniocerebral trauma']": 1882, "['shortness of breath', 'short stature']": 1858, "['palpitations']": 1804, "['ex

## Evaluation

In [56]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

labels = ["Low", "Medium", "High"]

print("Train accuracy:", accuracy_score(train_processed["risk"], train_processed["pred_risk"]))
print("Test accuracy:", accuracy_score(test_processed["risk"], test_processed["pred_risk"]))
print("Test macro F1:", f1_score(test_processed["risk"], test_processed["pred_risk"], average="macro"))

print("\nClassification report:")
print(classification_report(test_processed["risk"], test_processed["pred_risk"], labels=labels))

print("\nConfusion matrix:")
print(confusion_matrix(test_processed["risk"], test_processed["pred_risk"], labels=labels))

errors = test_processed[test_processed["risk"] != test_processed["pred_risk"]][
    ["chiefcomplaint", "affirmed_concepts", "risk", "pred_risk"]
]

errors.head(30)

Train accuracy: 0.38249926748907215
Test accuracy: 0.3848380114091632
Test macro F1: 0.37191151540192746

Classification report:
              precision    recall  f1-score   support

         Low       0.11      0.83      0.20      5920
      Medium       0.71      0.37      0.48     45012
        High       0.63      0.33      0.43     32685

    accuracy                           0.38     83617
   macro avg       0.48      0.51      0.37     83617
weighted avg       0.64      0.38      0.44     83617


Confusion matrix:
[[ 4915   928    77]
 [22352 16508  6152]
 [15978  5951 10756]]


,chiefcomplaint,affirmed_concepts,risk,pred_risk
0,L Hip pain,['hip joint pain'],High,Low
2,ETOH,['ethanol'],Medium,Low
3,R Breast pain,['mastodynia'],Medium,Low
4,Headache,['headache'],High,Medium
5,DEPRESSION,['mental depression'],High,Low
10,"Femur fracture, s/p Fall, Transfer",['femoral fractures'],Medium,Low
12,ALLERGIC RXN,['allergic'],High,Low
13,Rectal pain,['rectal pain'],Medium,Low
14,ST,[],Medium,Low
16,R Ankle pain,['ankle pain'],Medium,Low


In [57]:
from collections import Counter

missed_high = test_processed[(test_processed["risk"] == "High") & (test_processed["pred_risk"] != "High")]
counter = Counter()

for x in missed_high["affirmed_concepts"]:
    counter.update(parse_concepts(x))

counter.most_common(30)

[('abdominal pain', 1815),
 ('fever', 1061),
 ('abnormal', 996),
 ('mental state', 983),
 ('headache', 837),
 ('dizziness', 800),
 ('vomiting', 767),
 ('nausea', 763),
 ('laboratory test finding', 536),
 ('back pain', 508),
 ('transfer - product ownership', 468),
 ('nausea and vomiting', 462),
 ('hematochezia', 392),
 ('palpitations', 367),
 ('hypersensitivity', 362),
 ('ethanol', 320),
 ('surgical transfer - action', 312),
 ('hypotension', 312),
 ('confusion', 303),
 ('mental depression', 269),
 ('diarrhea', 264),
 ('cough', 233),
 ('anemia', 222),
 ('hyperglycemia', 222),
 ('psychiatric problem', 222),
 ('tachycardia', 220),
 ('changing', 208),
 ('visual', 207),
 ('craniocerebral trauma', 177),
 ('hypoxia', 167)]